# 6. Tối ưu hoá truy vấn trên lược đồ hình sao (Star Schema Optimization)

Theo sách, Chương 19, mục 19.8 (ví dụ tối ưu truy vấn trong kho dữ liệu).

**Lược đồ hình sao** gồm một **bảng sự kiện (fact table)** rất lớn ở giữa, chứa các số đo (ví dụ số giờ làm) và khoá ngoại,
bao quanh là các **bảng chiều (dimension table)** nhỏ mô tả ngữ cảnh (ai, dự án nào, phòng nào, khi nào).

```
                 DIM_TIME (24)
                      |
DIM_DEPARTMENT (20) — FACT_WORK (~190.000) — DIM_PROJECT (500)
                      |
                 DIM_EMPLOYEE (10.000)
```

**Vì sao cần kỹ thuật riêng?** Truy vấn kho dữ liệu thường lọc trên **nhiều bảng chiều cùng lúc**. Mỗi điều kiện riêng lẻ chọn lọc **yếu**
(ví dụ 1/7 số dự án, 1/8 số tháng), nhưng **kết hợp lại** thì chọn lọc **rất mạnh**. Kế hoạch thông thường (nối bảng sự kiện lần lượt với từng bảng chiều)
vẫn phải đọc gần hết bảng sự kiện. Giữa các bảng chiều lại không có điều kiện nối, nên quy tắc kinh nghiệm "tránh tích Descartes" bỏ qua cách nhân các bảng chiều với nhau trước.

**Mục tiêu**

| Thí nghiệm | Kế hoạch | Ý chính |
|---|---|---|
| 1 | Hash join thông thường | Quét toàn bộ bảng sự kiện |
| 2 | Dùng chỉ mục của MỘT chiều chọn lọc nhất | Đọc ít hơn, nhưng vẫn đọc thừa nhiều dòng |
| 3 | **Biến đổi hình sao (star transformation)** với bitmap index | Chỉ đọc đúng các dòng thoả **mọi** điều kiện |
| 4 | **Loại bỏ nối ngược (join-back elimination)** | Không nối lại bảng chiều khi không cần cột của nó |
| 5 | **Nối hình sao (star join)**: tích Descartes các chiều + chỉ mục ghép | Rẻ nhất khi các chiều đã lọc rất nhỏ |
| 6 | Khi nào các kỹ thuật này **không** có lợi | Bộ tối ưu dựa trên chi phí phải chọn |

Chi phí đo bằng **số dòng phải đọc** từ bảng sự kiện và bảng chiều, số lần tra chỉ mục và số phép toán bitmap.
Phần PostgreSQL ở file `6.2_star_schema_postgres_demo.sql`. Notebook chỉ dùng Python chuẩn.

## 1. Dữ liệu: lược đồ hình sao dựng từ CSDL COMPANY

Bảng sự kiện `FACT_WORK` ghi **số giờ mỗi nhân viên làm cho mỗi dự án trong mỗi tháng**. Dữ liệu được làm lệch giống file `02_generate_large_data_pgadmin.sql`:
40% nhân viên thuộc phòng 5 (Research), 20% thuộc phòng 4, còn lại rải đều; dự án số nhỏ đông người hơn.

In [1]:
import random
from collections import defaultdict
from itertools import product

random.seed(42)
LOCATIONS = ["Houston", "Bellaire", "Sugarland", "Stafford", "Spring", "Humble", "Katy"]
LAST = ["Smith", "Wong", "Zelaya", "Wallace", "Narayan", "English", "Jabbar", "Borg", "Brown", "Davis"]
BOOK_DEPT = {1: "Headquarters", 4: "Administration", 5: "Research"}

DIM_DEPARTMENT = [{"dept_key": d, "dname": BOOK_DEPT.get(d, f"Dept_{d:02d}")} for d in range(1, 21)]
DIM_PROJECT = [{"proj_key": p, "pname": f"Project_{p:03d}", "plocation": random.choice(LOCATIONS)}
               for p in range(1, 501)]
DIM_TIME = [{"time_key": y * 100 + m, "year": y, "quarter": (m - 1) // 3 + 1, "month": m}
            for y in (2023, 2024) for m in range(1, 13)]


def random_dept():
    r = random.random()
    return 5 if r < 0.40 else 4 if r < 0.60 else random.randint(1, 20)


DIM_EMPLOYEE = [{"emp_key": e, "lname": random.choice(LAST), "sex": random.choice("MF"), "dno": random_dept()}
                for e in range(1, 10_001)]

FACT_WORK = []
for e in DIM_EMPLOYEE:
    projects = {1 + int(500 * random.random() ** 2) for _ in range(random.randint(1, 3))}
    for p in sorted(projects):
        for t in DIM_TIME:
            if random.random() < 0.4:                    # không phải tháng nào cũng làm
                FACT_WORK.append({"emp_key": e["emp_key"], "proj_key": p, "dept_key": e["dno"],
                                  "time_key": t["time_key"], "hours": round(random.uniform(1, 40), 1)})

DIMS = {"department": DIM_DEPARTMENT, "project": DIM_PROJECT, "time": DIM_TIME, "employee": DIM_EMPLOYEE}
KEY = {"department": "dept_key", "project": "proj_key", "time": "time_key", "employee": "emp_key"}

for name, rows in DIMS.items():
    print(f"DIM_{name.upper():<11}: {len(rows):>7,} dòng")
print(f"FACT_WORK      : {len(FACT_WORK):>7,} dòng")

DIM_DEPARTMENT :      20 dòng
DIM_PROJECT    :     500 dòng
DIM_TIME       :      24 dòng
DIM_EMPLOYEE   :  10,000 dòng
FACT_WORK      : 192,218 dòng


## 2. Chỉ mục trên khoá ngoại của bảng sự kiện

- **Chỉ mục B-tree** (mô phỏng bằng `dict`): giá trị khoá → danh sách vị trí dòng.
- **Bitmap index**: với mỗi giá trị khoá, một chuỗi bit dài bằng số dòng của bảng sự kiện, bit thứ i = 1 nếu dòng i có giá trị đó.
  Dùng số nguyên Python làm chuỗi bit, nên phép OR / AND trên cả chuỗi chỉ là `|` và `&`.
  Bitmap index rất gọn với cột có ít giá trị khác nhau, đúng như khoá ngoại của bảng sự kiện.
- **Chỉ mục ghép** trên `(dept_key, time_key, proj_key)` dùng cho thí nghiệm 5.

In [2]:
BTREE = {col: defaultdict(list) for col in KEY.values()}
BITMAP = {col: defaultdict(int) for col in KEY.values()}
COMPOSITE = defaultdict(list)
for i, f in enumerate(FACT_WORK):
    for col in KEY.values():
        BTREE[col][f[col]].append(i)
        BITMAP[col][f[col]] |= 1 << i
    COMPOSITE[(f["dept_key"], f["time_key"], f["proj_key"])].append(i)


def bit_positions(bm):
    # Vị trí các bit 1 trong chuỗi bit
    while bm:
        low = bm & -bm
        yield low.bit_length() - 1
        bm ^= low


for col, bms in BITMAP.items():
    print(f"Bitmap index trên {col:<9}: {len(bms):>6,} chuỗi bit, mỗi chuỗi {len(FACT_WORK):,} bit")

Bitmap index trên dept_key :     20 chuỗi bit, mỗi chuỗi 192,218 bit
Bitmap index trên proj_key :    500 chuỗi bit, mỗi chuỗi 192,218 bit
Bitmap index trên time_key :     24 chuỗi bit, mỗi chuỗi 192,218 bit
Bitmap index trên emp_key  : 10,000 chuỗi bit, mỗi chuỗi 192,218 bit


## 3. Câu truy vấn kho dữ liệu

Tổng số giờ theo dự án, của phòng *Dept_17*, cho các dự án đặt tại *Stafford*, trong quý 1 năm 2024.

```sql
SELECT p.pname, SUM(f.hours)
FROM   fact_work f, dim_department d, dim_project p, dim_time t
WHERE  f.dept_key = d.dept_key AND f.proj_key = p.proj_key AND f.time_key = t.time_key
  AND  d.dname = 'Dept_17' AND p.plocation = 'Stafford' AND t.year = 2024 AND t.quarter = 1
GROUP  BY p.pname;
```

Ô dưới đây đo độ chọn lọc của từng điều kiện riêng lẻ trên bảng sự kiện và khi kết hợp.

In [3]:
QUERY = {
    "department": lambda d: d["dname"] == "Dept_17",
    "project":    lambda p: p["plocation"] == "Stafford",
    "time":       lambda t: t["year"] == 2024 and t["quarter"] == 1,
}


def matching_keys(filters):
    return {dim: {r[KEY[dim]] for r in DIMS[dim] if pred(r)} for dim, pred in filters.items()}


def fact_fraction(filters):
    keys = matching_keys(filters)
    n = sum(all(f[KEY[d]] in keys[d] for d in filters) for f in FACT_WORK)
    return n


N = len(FACT_WORK)
for dim in QUERY:
    n = fact_fraction({dim: QUERY[dim]})
    print(f"Chỉ lọc trên DIM_{dim.upper():<11}: {n:>7,} dòng sự kiện ({n / N:6.2%})")
n_all = fact_fraction(QUERY)
print(f"Kết hợp cả ba điều kiện       : {n_all:>7,} dòng sự kiện ({n_all / N:6.2%})")

Chỉ lọc trên DIM_DEPARTMENT :   3,833 dòng sự kiện ( 1.99%)
Chỉ lọc trên DIM_PROJECT    :  20,543 dòng sự kiện (10.69%)
Chỉ lọc trên DIM_TIME       :  24,069 dòng sự kiện (12.52%)
Kết hợp cả ba điều kiện       :      62 dòng sự kiện ( 0.03%)


**Nhận xét:** từng điều kiện riêng lẻ giữ lại 2% đến 13% bảng sự kiện, nhưng kết hợp cả ba chỉ còn khoảng **0,03%**.
Một kế hoạch tốt phải tận dụng **cùng lúc** cả ba điều kiện **trước khi** đọc bảng sự kiện.

Mọi kế hoạch bên dưới đều ghi chi phí vào một bộ đếm chung, và trả về kết quả để kiểm tra các kế hoạch cho cùng đáp án.

In [4]:
def new_cost():
    return {"Dòng sự kiện đọc": 0, "Dòng chiều đọc": 0, "Lần tra chỉ mục/băm": 0, "Phép toán bitmap": 0}


def filter_dims(filters, cost):
    # Đọc từng bảng chiều, giữ các dòng thoả điều kiện (dạng bảng băm theo khoá)
    kept = {}
    for dim, pred in filters.items():
        cost["Dòng chiều đọc"] += len(DIMS[dim])
        kept[dim] = {r[KEY[dim]]: r for r in DIMS[dim] if pred(r)}
    return kept


def aggregate(rows, project_rows, cost):
    # Nối ngược (join back) với DIM_PROJECT để lấy pname, rồi cộng số giờ
    out = defaultdict(float)
    for f in rows:
        cost["Lần tra chỉ mục/băm"] += 1
        out[project_rows[f["proj_key"]]["pname"]] += f["hours"]
    return {k: round(v, 1) for k, v in out.items()}


RESULTS, COSTS = {}, {}


def report(name, result, cost):
    RESULTS[name], COSTS[name] = result, cost
    print(f"{name}: {len(result)} nhóm kết quả")
    for k, v in cost.items():
        print(f"   {k:<22}{v:>12,}")

## 4. Thí nghiệm 1: Hash join thông thường

Dựng bảng băm cho từng bảng chiều đã lọc, rồi **quét toàn bộ** bảng sự kiện, với mỗi dòng tra vào các bảng băm.
Đây là kế hoạch PostgreSQL chọn khi bảng sự kiện chưa có chỉ mục trên khoá ngoại.

In [5]:
def plan_hash_join(filters):
    cost = new_cost()
    kept = filter_dims(filters, cost)
    rows = []
    for f in FACT_WORK:
        cost["Dòng sự kiện đọc"] += 1
        ok = True
        for dim in filters:
            cost["Lần tra chỉ mục/băm"] += 1
            if f[KEY[dim]] not in kept[dim]:
                ok = False
                break
        if ok:
            rows.append(f)
    return aggregate(rows, kept["project"], cost), cost


report("1. Hash join thông thường", *plan_hash_join(QUERY))

1. Hash join thông thường: 32 nhóm kết quả
   Dòng sự kiện đọc           192,218
   Dòng chiều đọc                 544
   Lần tra chỉ mục/băm        196,567
   Phép toán bitmap                 0


## 5. Thí nghiệm 2: Dùng chỉ mục của một chiều

Bộ tối ưu thông thường chọn **một** chiều chọn lọc nhất (DIM_DEPARTMENT, 2%), tra chỉ mục B-tree trên `fact_work.dept_key`
để lấy các dòng sự kiện của phòng đó, rồi mới kiểm tra hai điều kiện còn lại.

In [6]:
def plan_one_index(filters, driving="department"):
    cost = new_cost()
    kept = filter_dims(filters, cost)
    rows = []
    for k in kept[driving]:
        cost["Lần tra chỉ mục/băm"] += 1
        for i in BTREE[KEY[driving]].get(k, []):
            f = FACT_WORK[i]
            cost["Dòng sự kiện đọc"] += 1
            cost["Lần tra chỉ mục/băm"] += len(filters) - 1       # kiểm tra các chiều còn lại
            if all(f[KEY[d]] in kept[d] for d in filters if d != driving):
                rows.append(f)
    return aggregate(rows, kept["project"], cost), cost


report("2. Chỉ mục một chiều", *plan_one_index(QUERY))

2. Chỉ mục một chiều: 32 nhóm kết quả
   Dòng sự kiện đọc             3,833
   Dòng chiều đọc                 544
   Lần tra chỉ mục/băm          7,729
   Phép toán bitmap                 0


## 6. Thí nghiệm 3: Biến đổi hình sao (Star Transformation)

Bộ tối ưu **viết lại** câu truy vấn: thay các phép nối bằng các truy vấn con trên bảng chiều.

```sql
SELECT f.proj_key, f.hours FROM fact_work f
WHERE  f.dept_key IN (SELECT dept_key FROM dim_department WHERE dname = 'Dept_17')
  AND  f.proj_key IN (SELECT proj_key FROM dim_project WHERE plocation = 'Stafford')
  AND  f.time_key IN (SELECT time_key FROM dim_time WHERE year = 2024 AND quarter = 1);
```

Cách thực hiện:

1. Chạy các truy vấn con trên bảng chiều (nhỏ, rẻ) để có tập khoá.
2. Với mỗi chiều: **OR** các chuỗi bit của mọi khoá trong tập → một chuỗi bit cho chiều đó.
3. **AND** chuỗi bit của các chiều → chuỗi bit cuối, bit 1 là dòng thoả **mọi** điều kiện.
4. Chỉ đọc các dòng sự kiện có bit 1.
5. **Nối ngược (join back)** với bảng chiều để lấy các cột cần hiển thị (`pname`).

In [7]:
def plan_star_transformation(filters, join_back=True, verbose=False):
    cost = new_cost()
    kept = filter_dims(filters, cost)                                   # bước 1
    final = None
    for dim in filters:
        bm = 0
        for k in kept[dim]:                                             # bước 2: OR
            bm |= BITMAP[KEY[dim]].get(k, 0)
            cost["Phép toán bitmap"] += 1
        if verbose:
            print(f"   Bitmap DIM_{dim.upper():<11}: {len(kept[dim]):>4} khoá -> {bin(bm).count('1'):>7,} bit 1")
        if final is None:
            final = bm
        else:
            final &= bm                                                 # bước 3: AND
            cost["Phép toán bitmap"] += 1
    if verbose:
        print(f"   {'Sau khi AND':<22}: {'':>9} -> {bin(final).count('1'):>7,} bit 1")
    rows = [FACT_WORK[i] for i in bit_positions(final)]                 # bước 4
    cost["Dòng sự kiện đọc"] += len(rows)
    if not join_back:
        return {"Tổng số giờ": round(sum(f["hours"] for f in rows), 1)}, cost
    return aggregate(rows, kept["project"], cost), cost                 # bước 5


report("3. Biến đổi hình sao", *plan_star_transformation(QUERY, verbose=True))

   Bitmap DIM_DEPARTMENT :    1 khoá ->   3,833 bit 1
   Bitmap DIM_PROJECT    :   61 khoá ->  20,543 bit 1
   Bitmap DIM_TIME       :    3 khoá ->  24,069 bit 1
   Sau khi AND           :           ->      62 bit 1
3. Biến đổi hình sao: 32 nhóm kết quả
   Dòng sự kiện đọc                62
   Dòng chiều đọc                 544
   Lần tra chỉ mục/băm             62
   Phép toán bitmap                67


## 7. Thí nghiệm 4: Loại bỏ nối ngược (Join-back Elimination)

Nếu câu truy vấn **không cần cột nào** của bảng chiều ngoài khoá (ví dụ chỉ hỏi *tổng số giờ*), bước 5 là thừa:
khoá đã có sẵn trong bảng sự kiện và điều kiện của chiều đã được kiểm tra qua bitmap. Bộ tối ưu bỏ phép nối ngược.

```sql
SELECT SUM(f.hours) FROM fact_work f, dim_department d, dim_project p, dim_time t
WHERE  ... (như trên) ...;
```

In [8]:
_, cost_jb = plan_star_transformation(QUERY, join_back=True)
total_jb = round(sum(RESULTS["3. Biến đổi hình sao"].values()), 1)
res_no_jb, cost_no_jb = plan_star_transformation(QUERY, join_back=False)

print(f"Có nối ngược     : tổng số giờ = {total_jb}, tra bảng chiều {cost_jb['Lần tra chỉ mục/băm']} lần")
print(f"Bỏ nối ngược     : tổng số giờ = {res_no_jb['Tổng số giờ']}, tra bảng chiều {cost_no_jb['Lần tra chỉ mục/băm']} lần")
print("Cùng kết quả:", abs(total_jb - res_no_jb["Tổng số giờ"]) < 0.05)

Có nối ngược     : tổng số giờ = 1296.8, tra bảng chiều 62 lần
Bỏ nối ngược     : tổng số giờ = 1296.8, tra bảng chiều 0 lần
Cùng kết quả: True


## 8. Thí nghiệm 5: Nối hình sao (Star Join) bằng tích Descartes các chiều

Cách thứ hai: **nhân các bảng chiều đã lọc với nhau trước** (tích Descartes), mỗi tổ hợp khoá `(dept, time, proj)`
tra **một lần** vào chỉ mục ghép trên bảng sự kiện. Kế hoạch này vi phạm quy tắc "tránh tích Descartes" của notebook 1,
nhưng rẻ khi các chiều sau khi lọc đều rất nhỏ: ở đây chỉ `1 × 3 × số dự án ở Stafford` tổ hợp.

In [9]:
def plan_star_join(filters):
    cost = new_cost()
    kept = filter_dims(filters, cost)
    order = ["department", "time", "project"]                    # đúng thứ tự cột của chỉ mục ghép
    combos = list(product(*(kept[d] for d in order)))
    rows = []
    for combo in combos:
        cost["Lần tra chỉ mục/băm"] += 1
        for i in COMPOSITE.get(combo, []):
            rows.append(FACT_WORK[i])
            cost["Dòng sự kiện đọc"] += 1
    print(f"   Tích Descartes các chiều đã lọc: {' × '.join(str(len(kept[d])) for d in order)} = {len(combos)} tổ hợp")
    return aggregate(rows, kept["project"], cost), cost


report("5. Nối hình sao", *plan_star_join(QUERY))

   Tích Descartes các chiều đã lọc: 1 × 3 × 61 = 183 tổ hợp
5. Nối hình sao: 32 nhóm kết quả
   Dòng sự kiện đọc                62
   Dòng chiều đọc                 544
   Lần tra chỉ mục/băm            245
   Phép toán bitmap                 0


## 9. So sánh

In [10]:
first = next(iter(RESULTS.values()))
same = all(r == first for r in RESULTS.values())
print(f"Cả {len(RESULTS)} kế hoạch cho cùng kết quả: {same}\n")

cols = list(new_cost())
print(f"{'Kế hoạch':<28}" + "".join(f"{c:>22}" for c in cols))
for name, cost in COSTS.items():
    print(f"{name:<28}" + "".join(f"{cost[c]:>22,}" for c in cols))

base = COSTS["1. Hash join thông thường"]["Dòng sự kiện đọc"]
print()
for name, cost in COSTS.items():
    print(f"{name:<28} đọc {cost['Dòng sự kiện đọc'] / base:7.2%} bảng sự kiện")

Cả 4 kế hoạch cho cùng kết quả: True

Kế hoạch                          Dòng sự kiện đọc        Dòng chiều đọc   Lần tra chỉ mục/băm      Phép toán bitmap
1. Hash join thông thường                  192,218                   544               196,567                     0
2. Chỉ mục một chiều                         3,833                   544                 7,729                     0
3. Biến đổi hình sao                            62                   544                    62                    67
5. Nối hình sao                                 62                   544                   245                     0

1. Hash join thông thường    đọc 100.00% bảng sự kiện
2. Chỉ mục một chiều         đọc   1.99% bảng sự kiện
3. Biến đổi hình sao         đọc   0.03% bảng sự kiện
5. Nối hình sao              đọc   0.03% bảng sự kiện


**Nhận xét:**

- **Hash join thông thường** phải đọc toàn bộ bảng sự kiện, dù chỉ khoảng 0,03% số dòng có ích.
- **Chỉ mục một chiều** đọc ít hơn nhiều, nhưng vẫn phải đọc mọi dòng của phòng Dept_17 rồi loại phần lớn ở bước kiểm tra sau.
- **Biến đổi hình sao** đọc **đúng** các dòng có trong kết quả: việc lọc diễn ra hoàn toàn trên bitmap, trước khi chạm tới bảng sự kiện.
  Các phép OR / AND trên bitmap rất rẻ so với đọc dòng từ đĩa.
- **Nối hình sao** cũng đọc đúng các dòng cần, với số lần tra chỉ mục bằng số tổ hợp của tích Descartes.

## 10. Thí nghiệm 6: Khi nào các kỹ thuật này không có lợi?

Hai kỹ thuật trên không phải lúc nào cũng tốt hơn. Thử với hai câu truy vấn khác:

- **Q_yếu**: điều kiện chọn lọc yếu (phòng Research 40%, cả năm 2024, mọi dự án **không** ở Stafford): kết hợp lại vẫn giữ nhiều dòng.
  Đọc các dòng rải rác theo bitmap còn chậm hơn quét tuần tự, vì mỗi dòng có thể nằm ở một khối đĩa khác nhau.
- **Q_nhân_viên**: câu truy vấn chính, thêm điều kiện trên DIM_EMPLOYEE (`sex = 'F'`, khoảng 5.000 nhân viên).
  Tích Descartes các chiều **bùng nổ**, nhưng biến đổi hình sao vẫn ổn.

In [11]:
Q_WEAK = {
    "department": lambda d: d["dname"] == "Research",
    "project":    lambda p: p["plocation"] != "Stafford",
    "time":       lambda t: t["year"] == 2024,
}
Q_EMP = dict(QUERY, employee=lambda e: e["sex"] == "F")


def star_join_combos(filters):
    keys = matching_keys(filters)
    n = 1
    for d in filters:
        n *= len(keys[d])
    return n


print(f"{'Truy vấn':<14}{'Dòng kết quả':>14}{'Tỉ lệ bảng sự kiện':>22}{'Số tổ hợp star join':>22}")
for name, q in [("Q chính", QUERY), ("Q_yếu", Q_WEAK), ("Q_nhân_viên", Q_EMP)]:
    n = fact_fraction(q)
    print(f"{name:<14}{n:>14,}{n / N:>22.2%}{star_join_combos(q):>22,}")
print(f"\nSố dòng bảng sự kiện: {N:,}")

Truy vấn        Dòng kết quả    Tỉ lệ bảng sự kiện   Số tổ hợp star join
Q chính                   62                 0.03%                   183
Q_yếu                 36,196                18.83%                 5,268


Q_nhân_viên               29                 0.02%               914,451

Số dòng bảng sự kiện: 192,218


**Nhận xét:**

- Với **Q_yếu**, khoảng 1/5 bảng sự kiện thoả điều kiện. Biến đổi hình sao vẫn phải đọc từng ấy dòng, nhưng theo thứ tự rải rác,
  nên **quét tuần tự + hash join** (thí nghiệm 1) thường rẻ hơn. Số tổ hợp star join cũng lớn.
- Với **Q_nhân_viên**, số tổ hợp của tích Descartes **lớn hơn cả số dòng của bảng sự kiện**: nối hình sao trở thành kế hoạch tệ nhất.
  Biến đổi hình sao chỉ thêm một bitmap nữa để AND, không bị ảnh hưởng.
- Vì vậy các hệ quản trị như Oracle **không** áp dụng biến đổi hình sao một cách máy móc: bộ tối ưu **ước lượng chi phí** (notebook 2)
  của kế hoạch đã biến đổi và kế hoạch gốc, rồi chọn kế hoạch rẻ hơn.

## 11. Đối chiếu trên PostgreSQL

Mở file `6.2_star_schema_postgres_demo.sql` trong pgAdmin (CSDL `company`, đã chạy file 01 và 02). File tự dựng lược đồ hình sao `dw` với khoảng 1,2 triệu dòng sự kiện.

| Thí nghiệm trong notebook | Demo trong file SQL | Cần chỉ ra trong `EXPLAIN` |
|---|---|---|
| 1. Hash join thông thường | Demo 1 | `Seq Scan on fact_work`, rows ≈ 1,2 triệu |
| 2. Chỉ mục một chiều | Demo 2 | `Bitmap Index Scan on fact_work_dept_key_idx`, khoảng 24.850 dòng sự kiện |
| 3. Biến đổi hình sao | Demo 3 | `BitmapAnd` của 3 `Bitmap Index Scan`, chỉ khoảng 460 dòng sự kiện |
| 4. Bỏ nối ngược | Demo 4 | không còn nút `Hash Join` với `dim_project` |
| 5. Nối hình sao | Demo 5 | `Nested Loop` các bảng chiều, `Index Scan using fact_dtp`, loops = số tổ hợp |

Lưu ý: PostgreSQL **không có** bitmap index lưu trên đĩa và **không tự** thực hiện biến đổi hình sao như Oracle.
Nó dựng bitmap trong bộ nhớ từ chỉ mục B-tree lúc chạy (`Bitmap Index Scan`) và có thể AND chúng (`BitmapAnd`), nên ta tự viết lại câu truy vấn để mô phỏng.

> *(Dán ảnh kết quả EXPLAIN vào đây khi viết báo cáo)*

## 12. Kết luận

1. Truy vấn trên lược đồ hình sao lọc trên nhiều bảng chiều; từng điều kiện chọn lọc yếu nhưng kết hợp lại rất mạnh.
2. **Biến đổi hình sao** viết lại phép nối thành các truy vấn con `IN`, dùng bitmap index trên khoá ngoại của bảng sự kiện, OR trong từng chiều và AND giữa các chiều,
   nhờ đó chỉ đọc đúng các dòng sự kiện cần thiết, rồi mới **nối ngược** với bảng chiều để lấy cột hiển thị.
3. **Loại bỏ nối ngược** bỏ các phép nối không cần thiết khi câu truy vấn không dùng cột nào của bảng chiều.
4. **Nối hình sao** nhân các bảng chiều đã lọc rồi tra chỉ mục ghép, rất rẻ khi các chiều nhỏ nhưng bùng nổ khi một chiều còn nhiều dòng.
5. Không kỹ thuật nào luôn tốt nhất: hệ quản trị phải dùng **ước lượng chi phí** để quyết định có biến đổi hay không.